# Ming-3.0-audio-flash Cookbook

This notebook demonstrates direct Transformers inference for speech recognition and spoken question answering, followed by the equivalent vLLM-Omni API workflow. Run it from the repository root.

## 1. Environment

Install the direct-inference dependencies in an isolated environment before starting Jupyter:

```bash
python3 -m venv .venv-hf
source .venv-hf/bin/activate
pip install -r requirements-transformers.txt jupyter
```

In [ ]:
import os
from pathlib import Path

import torch
from IPython.display import Audio, display
from transformers import AutoModelForCausalLM, AutoProcessor

REPOSITORY_ROOT = Path.cwd().resolve()
MODEL_PATH = os.getenv("MODEL_PATH", "inclusionAI/Ming-3.0-audio-flash")
AUDIO_PATH = REPOSITORY_ROOT / "examples/audio/introduce_ant_group.wav"

if not AUDIO_PATH.is_file():
    raise FileNotFoundError(f"Run this notebook from the repository root: {AUDIO_PATH}")

display(Audio(filename=str(AUDIO_PATH)))

## 2. Load the model

`MODEL_PATH` may be a downloaded checkpoint directory or the Hugging Face model ID. The model is loaded in BF16 and distributed across visible GPUs automatically.

In [ ]:
processor = AutoProcessor.from_pretrained(
    MODEL_PATH, trust_remote_code=True, use_fast=False
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()

In [ ]:
def generate(messages, max_new_tokens=256):
    inputs = processor(messages).to(model.device)
    inputs["audio_feats"] = inputs["audio_feats"].to(model.dtype)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False
        )
    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    return processor.decode(generated_ids, skip_special_tokens=True).strip()

## 3. Speech recognition

In [ ]:
asr_messages = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": str(AUDIO_PATH)},
            {
                "type": "text",
                "text": "Please recognize the language of this speech and transcribe it, placing a ‘\\t’ between the identified language and the transcribe.",
            },
        ],
    }
]
print(generate(asr_messages))

## 4. Spoken question answering

In [ ]:
qa_messages = [
    {
        "role": "system",
        "content": (
            "You are a voice assistant. Answer questions or execute commands based on "
            "the audio content. Return plain text without Markdown."
        ),
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Answer the question or follow the instruction in the audio.",
            },
            {"type": "audio", "audio": str(AUDIO_PATH)},
        ],
    },
]
print(generate(qa_messages, max_new_tokens=512))

## 5. vLLM-Omni API

For production serving, start the text-output profile in a terminal:

```bash
MING_OUTPUT_MODE=text ./scripts/serve_vllm_omni.sh "$MODEL_PATH"
```

Then call it with the bundled client:

```bash
python examples/vllm_omni_audio_chat.py examples/audio/introduce_ant_group.wav --prompt "Answer the question or follow the instruction in the audio."
```

Use `MING_OUTPUT_MODE=audio` and `examples/vllm_omni_audio_streaming.py` to stream a spoken answer with latency metrics.